# Milestone 3: Song Search & Artist Recommendations
**Team Zyntra**

In this milestone, we tackled search and recommendations:
1. **Binary Search Tree (BST):** Index songs by title for $O(\log n)$ search and pruned prefix search returning titles in alphabetical order.
2. **Undirected Graph:** Model mutual artist relationships and recommend artists exactly 2 links away who are not already directly related, in alphabetical order.


In [1]:
from collections import defaultdict


class BSTNode:
    """Node in the BST storing title and songId."""
    def __init__(self, title, song_id):
        self.title = str(title)
        self.song_id = str(song_id)
        self.left = None
        self.right = None

    def __repr__(self):
        return f"BSTNode('{self.title}', '{self.song_id}')"


class SongSearchBST:
    """
    BST storing songs keyed by title.
    Supports exact find and pruned prefix search in alphabetical order.
    """
    def __init__(self):
        self.root = None
        self._size = 0

    def addSong(self, title, song_id):
        """Inserts song into BST keyed by title."""
        self.root = self._insert(self.root, str(title), str(song_id))
        self._size += 1

    def _insert(self, node, title, song_id):
        if node is None:
            return BSTNode(title, song_id)

        if title < node.title:
            node.left = self._insert(node.left, title, song_id)
        elif title > node.title:
            node.right = self._insert(node.right, title, song_id)
        else:
            # if title matches, update id
            node.song_id = song_id
        return node

    def findSong(self, title):
        """Exact title search in O(log n) average time."""
        curr = self.root
        target = str(title)
        while curr:
            if target == curr.title:
                return curr.song_id
            elif target < curr.title:
                curr = curr.left
            else:
                curr = curr.right
        return None

    def prefixSearch(self, prefix):
        """
        Returns matching titles in alphabetical order and skips subtrees that cannot match.
        """
        results = []
        p = str(prefix)
        if not p:
            return self.in_order_titles()

        self._prefix_search(self.root, p, results)
        return results

    def _prefix_search(self, node, prefix, results):
        if node is None:
            return

        L = len(prefix)
        node_prefix = node.title[:L]

        # 1. Prune left subtree if left keys can't match
        if prefix <= node_prefix:
            self._prefix_search(node.left, prefix, results)

        # 2. Check current node
        if node.title.startswith(prefix):
            results.append(node.title)

        # 3. Prune right subtree if right keys can't match
        if prefix >= node_prefix:
            self._prefix_search(node.right, prefix, results)

    def in_order_titles(self):
        titles = []
        def _in(node):
            if node:
                _in(node.left)
                titles.append(node.title)
                _in(node.right)
        _in(self.root)
        return titles

    def __len__(self):
        return self._size


class ArtistGraph:
    """
    Undirected graph representing mutual artist relationships.
    """
    def __init__(self):
        self.adj = defaultdict(set)

    def addRelation(self, artistA, artistB):
        """Adds mutual link between artistA and artistB."""
        a, b = str(artistA), str(artistB)
        if a == b:
            return  # skip self-loops
        self.adj[a].add(b)
        self.adj[b].add(a)

    def recommendArtists(self, artist):
        """
        Returns artists exactly two links away who are not already directly related,
        in alphabetical order.
        """
        target = str(artist)
        if target not in self.adj:
            return []

        direct_neighbors = self.adj[target]
        candidates = set()

        # explore 2-hop neighbors
        for neighbor in direct_neighbors:
            for second_degree in self.adj[neighbor]:
                # Filter:
                # 1. Not the artist themselves
                # 2. Not already directly related (1-hop)
                if second_degree != target and second_degree not in direct_neighbors:
                    candidates.add(second_degree)

        return sorted(list(candidates))


### Demo 1: Indexing & Exact Title Search
Testing BST insertion and exact find operations.


In [2]:
bst = SongSearchBST()

songs = [
    ("Hotel California", "S101"),
    ("Bohemian Rhapsody", "S102"),
    ("Stairway to Heaven", "S103"),
    ("Blinding Lights", "S104"),
    ("Circles", "S105"),
    ("Starboy", "S106"),
    ("Stay", "S107")
]

for title, sid in songs:
    bst.addSong(title, sid)

print("Total songs in BST:", len(bst))
print("Find 'Starboy':", bst.findSong("Starboy"))
print("Find 'Hotel California':", bst.findSong("Hotel California"))
print("Find 'Unknown Track':", bst.findSong("Unknown Track"))


Total songs in BST: 7
Find 'Starboy': S106
Find 'Hotel California': S101
Find 'Unknown Track': None


### Demo 2: Pruned Prefix Search (Alphabetical Order)
Testing prefix search with subtree skipping.


In [3]:
# Search for tracks starting with 'Sta'
matches = bst.prefixSearch("Sta")
print("Prefix query 'Sta' matches:")
print(matches)

# Search for tracks starting with 'B'
b_matches = bst.prefixSearch("B")
print("\nPrefix query 'B' matches:")
print(b_matches)

# Subtree pruning demonstration:
visited = []
def count_visited(node, prefix):
    if not node:
        return
    visited.append(node.title)
    L = len(prefix)
    node_pref = node.title[:L]
    if prefix <= node_pref:
        count_visited(node.left, prefix)
    if prefix >= node_pref:
        count_visited(node.right, prefix)

count_visited(bst.root, "Hotel")
print(f"\nSearching for 'Hotel' only visited {len(visited)} nodes out of {len(bst)} total nodes!")
print("Visited:", " -> ".join(visited))


Prefix query 'Sta' matches:
['Stairway to Heaven', 'Starboy', 'Stay']

Prefix query 'B' matches:
['Blinding Lights', 'Bohemian Rhapsody']

Searching for 'Hotel' only visited 4 nodes out of 7 total nodes!
Visited: Hotel California -> Bohemian Rhapsody -> Circles -> Stairway to Heaven


### Demo 3: Artist Graph & 2-Hop Recommendations
Testing artist relations and recommendation filtering (excluding direct links and triangles).


In [4]:
graph = ArtistGraph()

# Add mutual links
graph.addRelation("The Weeknd", "Daft Punk")
graph.addRelation("The Weeknd", "Travis Scott")
graph.addRelation("Daft Punk", "Pharrell")
graph.addRelation("Daft Punk", "Kanye")
graph.addRelation("Travis Scott", "Drake")
graph.addRelation("The Weeknd", "Drake")  # DIRECT LINK! Drake is already directly related to The Weeknd

print("Direct relationships for 'The Weeknd':", graph.adj["The Weeknd"])

# Request recommendations for 'The Weeknd'
recs = graph.recommendArtists("The Weeknd")
print("\nRecommendations for 'The Weeknd' (2 links away, not direct):")
print(recs)

# Notice Drake is excluded because Drake is directly connected, even though Weeknd -> Travis -> Drake is 2 hops!


Direct relationships for 'The Weeknd': {'Daft Punk', 'Drake', 'Travis Scott'}

Recommendations for 'The Weeknd' (2 links away, not direct):
['Kanye', 'Pharrell']


### Sanity Tests for Milestone 3


In [5]:
def run_m3_tests():
    tree = SongSearchBST()
    tree.addSong("Song C", "C3")
    tree.addSong("Song A", "A1")
    tree.addSong("Song B", "B2")
    tree.addSong("Song D", "D4")

    # 1. Exact find
    assert tree.findSong("Song B") == "B2"
    assert tree.findSong("Ghost") is None

    # 2. Prefix search
    tree.addSong("Sunflower", "SF")
    tree.addSong("Sunroof", "SR")
    tree.addSong("Sunday Morning", "SM")
    res = tree.prefixSearch("Sun")
    assert res == ["Sunday Morning", "Sunflower", "Sunroof"]

    # 3. Graph relations
    g = ArtistGraph()
    g.addRelation("Artist1", "Artist2")
    g.addRelation("Artist2", "Artist3")
    # Artist3 is 2 hops away from Artist1
    assert g.recommendArtists("Artist1") == ["Artist3"]

    # 4. Triangle: add direct link between 1 and 3
    g.addRelation("Artist1", "Artist3")
    # Now Artist3 is 1 hop away directly, so recommendations must be empty!
    assert g.recommendArtists("Artist1") == []

    # 5. Alphabetical check
    g.addRelation("Artist2", "ArtistZ")
    g.addRelation("Artist2", "ArtistM")
    assert g.recommendArtists("Artist1") == ["ArtistM", "ArtistZ"]

    print("All Milestone 3 tests passed successfully!")

run_m3_tests()


All Milestone 3 tests passed successfully!
